# AI Gateway PoC: hands-on lab

Validates each step of the deck. Use synthetic prompts only. Clear all outputs before sharing or committing this notebook.

## Checklist: create before the session
- [ ] Resource group (owner + expiry tags)
- [ ] Foundry account with one chat model deployment and quota
- [ ] Backend credential (provider key or managed identity + role)
- [ ] Application Insights / approved OpenTelemetry destination
- [ ] Safe remote MCP server or OpenAPI definition

## Create live in the gateway portal
- [ ] AI Gateway instance (copy runtime endpoint)
- [ ] Runtime keys: `app-a`, `app-b`
- [ ] Rate-limit policy counted by caller identity
- [ ] MCP server publishing one approved tool

In [ ]:
%pip install -q requests

## 0. Configuration
Values are typed at runtime and kept in memory only.
- Base URL example: `https://<gateway>.azure-api.net/default/models/openai/v1`
- Tool URL example: `https://<gateway>.azure-api.net/default/toolservers/<server-name>/mcp`

In [ ]:
from getpass import getpass

BASE_URL = input("Gateway model base URL: ").strip().rstrip("/")
MODEL = input("Registered gateway model name: ").strip()
KEY_A = getpass("Runtime key for app-a: ")
KEY_B = getpass("Runtime key for app-b: ")
TOOL_URL = input("MCP toolserver URL (blank to skip Lab 6): ").strip()
KEY_HEADER = "api-key"  # change if your gateway expects another header
print("Configured.")

In [ ]:
import requests

def chat(key, prompt="Explain an AI gateway briefly.", model=None):
    return requests.post(
        f"{BASE_URL}/chat/completions",
        headers={KEY_HEADER: key, "Content-Type": "application/json"},
        json={"model": model or MODEL, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )

## Lab 1: Governed access
Expect 200 with a reply. Then an invalid key should give 401 and a wrong model name should fail, not reroute.

In [ ]:
r = chat(KEY_A)
print("valid key:", r.status_code)
if r.ok:
    print(r.json()["choices"][0]["message"]["content"][:300])
    print("usage:", r.json().get("usage"))

print("invalid key:", chat("not-a-real-key").status_code, "(expect 401)")
print("wrong model:", chat(KEY_A, model="does-not-exist").status_code, "(expect 4xx)")

## Lab 2: Policy enforcement and isolated counters
Before running, add a small request or token rate limit in the portal, counted by caller identity. Exhaust app-a, then confirm app-b still has its own allowance.

In [ ]:
def burst(key, label, n=15):
    codes = []
    for _ in range(n):
        r = chat(key, "Say hi.")
        codes.append(r.status_code)
        if r.status_code == 429:
            print(f"{label}: 429, Retry-After={r.headers.get('Retry-After')}")
            break
    print(label, codes)
    return codes

a = burst(KEY_A, "app-a")
b = burst(KEY_B, "app-b", n=2)
assert 429 in a, "app-a was never limited: check the policy limit and scope"
assert b[0] == 200, "app-b should have its own allowance"
print("Counters are isolated per caller.")

## Lab 5: Telemetry
Generate successful and rejected traffic (above), then inspect it in Application Insights. Check logging settings first: prompts and responses may be sensitive.

In [ ]:
import time
time.sleep(60)  # allow the policy window to reset
for _ in range(3):
    u = chat(KEY_B, "One sentence on caching.").json().get("usage", {})
    print(u)

print("""
Query in Application Insights > Logs (adapt table/field names to your schema):

    dependencies
    | where timestamp > ago(30m)
    | extend model = tostring(customDimensions["gen_ai.request.model"]),
             inTok = toint(customDimensions["gen_ai.usage.input_tokens"]),
             outTok = toint(customDimensions["gen_ai.usage.output_tokens"])
    | summarize calls = count(), input = sum(inTok), output = sum(outTok),
                p95_ms = percentile(duration, 95) by model, resultCode
""")

## Lab 6: Approved MCP tool
Lists the toolserver's tools through the gateway. Confirm the inventory is exactly what you approved.

In [ ]:
import json

def parse(resp):
    if "text/event-stream" in resp.headers.get("Content-Type", ""):
        for line in resp.text.splitlines():
            if line.startswith("data:"):
                return json.loads(line[5:])
        return {}
    return resp.json() if resp.text else {}

def mcp(method, params=None, session=None, notify=False):
    headers = {KEY_HEADER: KEY_A, "Content-Type": "application/json",
               "Accept": "application/json, text/event-stream"}
    if session:
        headers["Mcp-Session-Id"] = session
    body = {"jsonrpc": "2.0", "method": method, "params": params or {}}
    if not notify:
        body["id"] = 1
    return requests.post(TOOL_URL, headers=headers, json=body, timeout=60)

if TOOL_URL:
    init = mcp("initialize", {"protocolVersion": "2025-03-26", "capabilities": {},
                              "clientInfo": {"name": "lab", "version": "1.0"}})
    sid = init.headers.get("Mcp-Session-Id")
    mcp("notifications/initialized", session=sid, notify=True)
    tools = parse(mcp("tools/list", session=sid)).get("result", {}).get("tools", [])
    for t in tools:
        print("-", t["name"], ":", t.get("description", "")[:80])
else:
    print("Skipped: no tool URL provided.")

In [ ]:
# Call one approved tool: set TOOL_NAME and ARGS from the listing above.
TOOL_NAME = ""
ARGS = {}
if TOOL_URL and TOOL_NAME:
    print(json.dumps(parse(mcp("tools/call", {"name": TOOL_NAME, "arguments": ARGS}, session=sid)), indent=2)[:1500])

## Acceptance evidence and cleanup
- [ ] Model call succeeds via the gateway; invalid key rejected; no provider key in the client
- [ ] Limits reject excess traffic; app-a and app-b counters are independent
- [ ] Token and latency telemetry visible
- [ ] Only the approved tool is exposed
- [ ] Revoke app-a/app-b keys, then delete the lab resource group
- [ ] Clear notebook outputs before saving